# UAE cancer workbook: end-to-end exploratory project

**Business question:** can diagnosis-time attributes classify the workbook's recorded `Outcome` label? This is a teaching benchmark, not a clinical model. The source, license, provenance, and outcome observation time are unverified. Do not use for patient care.

The supplied workbook has 200,000 rows and 20 fields. This notebook documents quality review, cleaning, EDA, feature design, fair model comparison, tuning, evaluation, and deployment handoff.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pipeline import DATA_PATH, FEATURES, clean_data, load_raw, train_model
print('Workbook:', DATA_PATH)

## 1. Data understanding and quality checks
The missing death date/cause values are expected for most non-deceased records and must not be imputed. Missing comorbidity means unknown, not absence. Check duplicate identifiers, types, ranges, and date logic before modeling.

In [ ]:
raw = load_raw()
print('Shape:', raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
display(raw.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Exact duplicate rows:', raw.duplicated().sum())
print('Duplicate patient IDs:', raw.Patient_ID.duplicated().sum())
display(raw.describe(include='all').T)

In [ ]:
for col in ['Age', 'Weight', 'Height']:
    print(col, raw[col].min(), raw[col].median(), raw[col].max())
print('Treatment before diagnosis:', (pd.to_datetime(raw.Treatment_Start_Date) < pd.to_datetime(raw.Diagnosis_Date)).sum())
display(raw.Outcome.value_counts(dropna=False).to_frame('records'))
display(pd.crosstab(raw.Cancer_Stage, raw.Outcome, normalize='index').round(3))

## 2. Cleaning and feature engineering
The reusable cleaner normalizes whitespace and data types, removes duplicate IDs and impossible broad-range measurements, preserves semantic missingness, and derives BMI. The observed workbook has no duplicates or range violations; thus cleaning does not invent quality problems. BMI units are assumed to be kg and cm and need provider confirmation.

In [ ]:
clean, quality = clean_data(raw)
display(quality)
print('Clean shape:', clean.shape)
print('BMI summary:')
display(clean.BMI.describe().to_frame())
display(clean.groupby('Cancer_Stage').BMI.median().to_frame('median_BMI'))

## 3. Exploratory analysis
Outcome proportions differ across recorded cancer stages in this file. That relationship may reflect synthetic label construction, selection, or real association; without provenance and observation-time definitions it cannot support clinical conclusions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=clean, x='Outcome', order=clean.Outcome.value_counts().index, ax=axes[0])
axes[0].set_title('Recorded outcome label counts'); axes[0].tick_params(axis='x', rotation=20)
stage_share = pd.crosstab(clean.Cancer_Stage, clean.Outcome, normalize='index')
stage_share.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Recorded outcome mix by cancer stage'); axes[1].set_ylabel('Share'); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=clean, x='Cancer_Type', hue='Outcome', ax=ax)
ax.set_title('Outcome labels across cancer type'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
print('Outcome by stage (row proportions):')
display(stage_share.round(3))

## 4. Model comparison, tuning, and held-out evaluation
The stratified test set is held out before training. Logistic regression provides an interpretable linear baseline; a decision tree tests nonlinear splits. Both use the same imputation and one-hot preprocessing. Macro-F1 weights each class equally. RandomizedSearchCV tunes regularization/class weighting on training folds only. These metrics are internal label-prediction performance and do not establish transportability or clinical benefit.

In [ ]:
model, metrics = train_model(clean, tune=True)
pd.DataFrame(metrics['comparison']).T


In [ ]:
print('Tuning:', metrics['tuning'])
print('Final test metrics:')
display(pd.Series({k:v for k,v in metrics['test_metrics'].items() if isinstance(v, (float,int))}))
print('Class-wise report:')
display(pd.DataFrame(metrics['test_metrics']['classification_report']).T)
ConfusionMatrixDisplay(confusion_matrix=metrics['test_metrics']['confusion_matrix'], display_labels=metrics['test_metrics']['class_order']).plot(cmap='Blues', xticks_rotation=30)
plt.tight_layout()

## 5. Deployment and business interpretation
`app.py` presents dataset/model metrics and a demonstration form using the same diagnosis-time predictors. Run `python train.py` to serialize the preprocessing/model pipeline and metrics, then `python -m streamlit run app.py`. The model's class scores are not calibrated patient risks.

**Business recommendation:** use this analysis to guide data governance and collection design, not care decisions. Verify provenance/license, define a meaningful fixed follow-up horizon, obtain representative real data and clinical oversight, assess subgroup performance, calibrate probabilities, and run prospective validation before considering any operational use. The supplied workbook is not currently eligible for a public deployment without rights confirmation.